In [8]:
translation_names = ['WRJTx', 'WRJTy', 'WRJTz']
rot_names = ['WRJRx', 'WRJRy', 'WRJRz']
joint_names = [
    "thumb_cmc_roll",
    "thumb_cmc_yaw",
    "thumb_cmc_pitch",
    "thumb_mcp",
    "thumb_ip",
    "index_mcp_roll",
    "index_mcp_pitch",
    "index_pip",
    "index_dip",
    "middle_mcp_roll",
    "middle_mcp_pitch",
    "middle_pip",
    "middle_dip",
    "ring_mcp_pitch",
    "ring_pip",
    "ring_dip",
    "pinky_mcp_pitch",
    "pinky_pip",
    "pinky_dip"
]

thumb = {
    "thumb_cmc_roll",
    "thumb_cmc_yaw",
    "thumb_cmc_pitch",
    "thumb_mcp",
    "thumb_ip"
}

In [9]:
import numpy as np
from collections import defaultdict
import scipy.spatial.transform as transform
import numpy as np
from scipy.spatial.transform import Rotation as R

grasp_poses =np.load('/home/guizhewei/guizhewei/bodex_dataset/grasp_poses_retargeted_bodex_bottle_1obj.npy', allow_pickle=True).item()
# grasp_poses =np.load('/home/ubuntu/Documents/DexYCB/grasp_poses_opt.npy', allow_pickle=True).item()
grasp_poses.keys()
# obj_idx = 3

dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219,

In [10]:
# grasp_poses[obj_idx].keys()


In [11]:
# grasp_poses[obj_idx]['target_object_name']

In [12]:
# grasp_poses[obj_idx]['robot_pose'][0]

In [13]:
import numpy as np
from scipy.spatial.transform import Rotation as R

def euler_to_rotation_matrix(euler_angles):
    rotation = R.from_euler('XYZ', euler_angles, degrees=False)
    return rotation.as_matrix()


def quaternion_to_rotation_matrix(quaternion):
    rotation = R.from_quat(quaternion)
    return rotation.as_matrix()


def object_pose_to_matrix(position, quaternion):
    """
    Converts object pose (position and quaternion) to a 4x4 transformation matrix.
    
    Parameters:
    - position: (3,) numpy.ndarray, position [x, y, z]
    - quaternion: (4,) numpy.ndarray, [x, y, z, w] quaternion
    
    Returns:
    - transformation_matrix: (4, 4) numpy.ndarray, corresponding transformation matrix
    """
    quaternion = np.concatenate([quaternion[1:4], quaternion[0:1]]) # wxyz-> xyzw
    rotation_matrix = quaternion_to_rotation_matrix(quaternion)
    transformation_matrix = np.eye(4)
    transformation_matrix[:3, :3] = rotation_matrix
    transformation_matrix[:3, 3] = position
    return transformation_matrix


def hand_pose_to_matrix(position, quaternion):
    """
    Converts object pose (position and quaternion) to a 4x4 transformation matrix.
    
    Parameters:
    - position: (3,) numpy.ndarray, position [x, y, z]
    - quaternion: (4,) numpy.ndarray, [x, y, z, w] quaternion
    
    Returns:
    - transformation_matrix: (4, 4) numpy.ndarray, corresponding transformation matrix
    """
    rotation_matrix = quaternion_to_rotation_matrix(quaternion)
    transformation_matrix = np.eye(4)
    transformation_matrix[:3, :3] = rotation_matrix
    transformation_matrix[:3, 3] = position
    return transformation_matrix

In [14]:
def get_obj_centric_pose_for_opt(grasp_poses, obj_idx):
    ''' Obj pose '''
    obj_pos = grasp_poses[obj_idx]['target_pose_world'][0].p
    obj_quat = grasp_poses[obj_idx]['target_pose_world'][0].q
    object_pose = object_pose_to_matrix(obj_pos, obj_quat)
    # print(f"Object Position: {obj_pos}, Object Quaternion: {obj_quat}")
    # print(f"Object Pose: {object_pose}")

    ''' Hand pose '''
    hand_pos = grasp_poses[obj_idx]['robot_pose'][0][:3]
    hand_euler = grasp_poses[obj_idx]['robot_pose'][0][3:6]
    hand_quat = transform.Rotation.from_euler('XYZ', hand_euler, degrees=False).as_quat()
    hand_6drot = transform.Rotation.from_euler('XYZ', hand_euler, degrees=False).as_matrix()
    hand_6drot = hand_6drot[:, :2].T.ravel().tolist()
    # print(f"Hand Position: {hand_pos}, Hand Quaternion: {hand_quat}")
    # print(f"Hand 6drot: {hand_6drot}")
    # print(grasp_poses[obj_idx]['robot_pose'][0])

    ''' object-centric '''
    W_T_O = object_pose
    W_T_H = hand_pose_to_matrix(hand_pos, hand_quat)

    O_T_H = np.linalg.inv(W_T_O) @ W_T_H
    print(O_T_H)
    t_oh  = O_T_H[:3, 3]
    R_oh  = O_T_H[:3, :3]

    euler_oh = R.from_matrix(R_oh).as_euler('XYZ', degrees=False)
    print(f"Object-Centric Hand Euler (XYZ, rad): {euler_oh}")
    print(f"Object-Centric Hand Position: {t_oh}")

    hand_6drot = R_oh
    hand_6drot = hand_6drot[:, :2].T.ravel().tolist()
    hand_pos = t_oh
    hand_euler = euler_oh

    return hand_pos, hand_euler, object_pose, hand_6drot

In [15]:
grasp_poses

{0: {'original_idx': 0,
  'target_object_name': 'core_bottle_1071fa4cddb2da2fc8724d5673a063a6',
  'grasp_type': 'force_closure',
  'scale_name': 'scale006',
  'grasp_idx': 0,
  'scene_scale': 1.0,
  'obj_scale': array([0.06, 0.06, 0.06]),
  'target_pose_world': [Pose([0, 0, 0], [1, 0, 0, 0])],
  'shadow_qpos': array([-0.13348357, -0.05431002, -0.09412572,  0.8259839 , -0.06901718,
          0.29734516,  0.47389144, -0.19145474,  0.3490172 ,  0.49145466,
          0.49145466, -0.24863735,  0.63764757,  0.21769889,  0.21769889,
          0.26288855,  0.4582895 ,  0.7222829 ,  0.7222829 ,  0.50306815,
         -0.26941693,  0.60375863,  0.15717188,  0.15717188,  0.48667994,
          1.0803841 , -0.14304759, -0.03920776,  0.06292731], dtype=float32),
  'robot_pose': [array([-1.19191095e-01, -3.75814699e-02, -6.39957115e-02, -2.93790847e-01,
           5.76076627e-01,  2.58586630e-02,  3.09080243e-01,  1.28905267e-01,
          -6.76396042e-02,  8.97429883e-01,  8.08603823e-01, -7.20125794

In [16]:
ycb_optimize_dataset_list = []

for obj_idx in grasp_poses.keys():
    hand_pos, hand_euler, object_pose, hand_6drot = get_obj_centric_pose_for_opt(grasp_poses, obj_idx)
    robot_pose_joint = grasp_poses[obj_idx]['robot_pose'][0]
    map_idx = [0, 5, 10, 15, 18, 1, 6, 11, 16, 2, 7, 12, 17, 3, 8, 13, 4, 9, 14]
    mapped_joint = [robot_pose_joint[6:][i] for i in map_idx]
    # print("mapped_joint", mapped_joint)
    robot_joint_dict = defaultdict(float)
    for i, joint_name in enumerate(joint_names):
        robot_joint_dict[joint_name] = mapped_joint[i]
    for i, name in enumerate(translation_names):
        robot_joint_dict[name] = hand_pos[i]
    for i, name in enumerate(rot_names):
        robot_joint_dict[name] = hand_euler[i]

    ycb_optimize_dataset = defaultdict(dict)
    ycb_optimize_dataset['qpos'] = robot_joint_dict
    print("obj_idx", obj_idx)
    print("ycb_optimize_dataset['qpos']", ycb_optimize_dataset['qpos'])
    ycb_optimize_dataset['object_code'] = grasp_poses[obj_idx]['target_object_name']
    ycb_optimize_dataset['object_pose'] = object_pose
    ycb_optimize_dataset['hand_rot6d'] = hand_6drot
    ycb_optimize_dataset['idx'] = obj_idx
    
    # 添加 scene_scale 和 obj_scale
    if 'scene_scale' in grasp_poses[obj_idx]:
        ycb_optimize_dataset['scene_scale'] = grasp_poses[obj_idx]['scene_scale']
    if 'obj_scale' in grasp_poses[obj_idx]:
        ycb_optimize_dataset['obj_scale'] = grasp_poses[obj_idx]['obj_scale']

    ycb_optimize_dataset_list.append(ycb_optimize_dataset)

ycb_optimize_dataset_list

[[ 0.83832595 -0.02168282  0.54473797 -0.1191911 ]
 [-0.13294603  0.96091166  0.24284589 -0.03758147]
 [-0.52871066 -0.27600476  0.80267454 -0.06399571]
 [ 0.          0.          0.          1.        ]]
Object-Centric Hand Euler (XYZ, rad): [-0.29379085  0.57607663  0.02585866]
Object-Centric Hand Position: [-0.1191911  -0.03758147 -0.06399571]
obj_idx 0
ycb_optimize_dataset['qpos'] defaultdict(<class 'float'>, {'thumb_cmc_roll': 0.30908024311065674, 'thumb_cmc_yaw': -0.7201257944107056, 'thumb_cmc_pitch': 0.00027083035092800856, 'thumb_mcp': -1.2577345371246338, 'thumb_ip': -1.0060618562459946, 'index_mcp_roll': 0.1289052665233612, 'index_mcp_pitch': 0.7443326711654663, 'index_pip': 0.6556017994880676, 'index_dip': 0.7317827285885812, 'middle_mcp_roll': -0.06763960421085358, 'middle_mcp_pitch': 0.6385555863380432, 'middle_pip': 0.8515149354934692, 'middle_dip': 0.9720042988657951, 'ring_mcp_pitch': 0.897429883480072, 'ring_pip': 0.7866870358586312, 'ring_dip': 0.7866870358586312, 'p

[defaultdict(dict,
             {'qpos': defaultdict(float,
                          {'thumb_cmc_roll': 0.30908024311065674,
                           'thumb_cmc_yaw': -0.7201257944107056,
                           'thumb_cmc_pitch': 0.00027083035092800856,
                           'thumb_mcp': -1.2577345371246338,
                           'thumb_ip': -1.0060618562459946,
                           'index_mcp_roll': 0.1289052665233612,
                           'index_mcp_pitch': 0.7443326711654663,
                           'index_pip': 0.6556017994880676,
                           'index_dip': 0.7317827285885812,
                           'middle_mcp_roll': -0.06763960421085358,
                           'middle_mcp_pitch': 0.6385555863380432,
                           'middle_pip': 0.8515149354934692,
                           'middle_dip': 0.9720042988657951,
                           'ring_mcp_pitch': 0.897429883480072,
                           'ring_pip': 0.78668

In [17]:
ycb_optimize_dataset_list

[defaultdict(dict,
             {'qpos': defaultdict(float,
                          {'thumb_cmc_roll': 0.30908024311065674,
                           'thumb_cmc_yaw': -0.7201257944107056,
                           'thumb_cmc_pitch': 0.00027083035092800856,
                           'thumb_mcp': -1.2577345371246338,
                           'thumb_ip': -1.0060618562459946,
                           'index_mcp_roll': 0.1289052665233612,
                           'index_mcp_pitch': 0.7443326711654663,
                           'index_pip': 0.6556017994880676,
                           'index_dip': 0.7317827285885812,
                           'middle_mcp_roll': -0.06763960421085358,
                           'middle_mcp_pitch': 0.6385555863380432,
                           'middle_pip': 0.8515149354934692,
                           'middle_dip': 0.9720042988657951,
                           'ring_mcp_pitch': 0.897429883480072,
                           'ring_pip': 0.78668

In [18]:
# store dict in a specified path as npy file
import os
import json
output_path = '/home/guizhewei/guizhewei/grasp_pose_dataset/unoptimized/dexycb_robot_joint_dict_1104_2300_omnihand_from_shadow.npy'
if not os.path.exists(os.path.dirname(output_path)):
    os.makedirs(os.path.dirname(output_path))
np.save(output_path, ycb_optimize_dataset_list, allow_pickle=True)